In [0]:
df = spark.table("accelerator.metadata.table_configs")

display(df)

In [0]:
# dq_df = spark.table("accelerator.metadata.data_quality_rules")
# dq_df.show()
# rules = dq_df.filter("table_name = 'product'")
# rules.show()

In [0]:
# rules = (
#     dq_df
#     .filter("table_name = 'product'")
#     .collect()
# )

# expectations = []

# for rule in rules:

#     if rule["rule_type"] == "not_null":
#         expectations.append(
#             f'@dlt.expect_or_drop("{rule["column_name"]}_not_null", '
#             f'"{rule["column_name"]} IS NOT NULL")'
#         )

#     elif rule["rule_type"] == "greater":
#         expectations.append(
#             f'@dlt.expect_or_drop("{rule["column_name"]}_gt", '
#             f'"{rule["column_name"]} > {rule["rule_value"]}")'
#         )

In [0]:
# silver_template = """
# import dlt

# {expectations}

# @dlt.table(
#     name="silver_{table_name}"
# )
# def silver_{table_name}():

#     return (
#         dlt.read_stream("bronze_{table_name}")
#         .dropDuplicates(["{primary_key}"])
#     )
# """


In [0]:
full_load_template  = """
import dlt

{expectations}

@dlt.table(
    name="silver_{table_name}"
)
def silver_{table_name}():

    return (
        dlt.read("bronze_{table_name}")
        .dropDuplicates(["{primary_key}"])
    )
"""

incremental_template   = """
import dlt

{expectations}

@dlt.table(
    name="silver_{table_name}"
)
def silver_{table_name}():

    return (
        dlt.read_stream("bronze_{table_name}")
        .dropDuplicates(["{primary_key}"])
    )
"""


cdc_template   = """
import dlt

{expectations}

@dlt.table(
    name="silver_{table_name}"
)
def silver_{table_name}():

    return (
        dlt.read_stream("bronze_{table_name}")
        .dropDuplicates(["{primary_key}"])
    )
"""


In [0]:
# for row in df.collect():
#     print(f"Row Print = ", row,end="\n")
#     tableName = row["table_name"]
#     dq_df = spark.table("accelerator.metadata.data_quality_rules")
#     # dq_df.show()
#     # rules = dq_df.filter("table_name = 'product'")
#     # rules.show()
#     rules = (
#     dq_df
#     .filter(f"table_name = '{tableName}'")
#     .collect()
#     )
#     print(f"TableName Print = ",tableName,end="\n")
#     print(f"Rules Print = ",rules,end="\n")
#     print()


In [0]:
expectations = []
for row in df.collect():
    tableName = row["table_name"]
    print(tableName)
    dq_df = spark.table("accelerator.metadata.data_quality_rules")
    # dq_df.show()
    # rules = dq_df.filter("table_name = 'product'")
    # rules.show()
    rules = (
    dq_df
    .filter(f"table_name = '{tableName}'")
    .collect()
    )
    for rule in rules:

        if rule["rule_type"] == "not_null":
            expectations.append(
                f'@dlt.expect_or_drop("{rule["column_name"]}_not_null", '
                f'"{rule["column_name"]} IS NOT NULL")'
            )

        elif rule["rule_type"] == "greater":
            expectations.append(
                f'@dlt.expect_or_drop("{rule["column_name"]}_gt", '
                f'"{rule["column_name"]} > {rule["rule_value"]}")'
            )

    load_type = row["load_type"]

    if load_type == "full":
        template = full_load_template

    elif load_type == "incremental":
        template = incremental_template
    
    elif load_type == "cdc":
        template = cdc_template

            # print(expectations)

    code = template.format(
                table_name=row["table_name"],
                primary_key = row["primary_key"],
                expectations="\n".join(expectations)
            
            )
    expectations = []
    
    # print(code)
    file_path = f"/Volumes/accelerator/metadata/generated_code/silver_{tableName}.py"

    with open(file_path, "w") as f:
        f.write(code)

    print(f"Generated: {file_path}")

In [0]:
# path = f"/Volumes/accelerator/metadata/generated/bronze_{table}.py"

In [0]:
# for row in df.collect():

#     table_name = row["table_name"]
#     load_type = row["load_type"]

#     if load_type == "full":
#         template = full_load_template

#     elif load_type == "incremental":
#         template = incremental_template

#     code = template.format(
#         table_name=row["table_name"],
#         primary_key = row["primary_key"],
#         expectations="\n".join(expectations)
#     )

#     file_path = f"/Volumes/accelerator/metadata/generated_code/silver_{table_name}.py"

#     with open(file_path, "w") as f:
#         f.write(code)

#     print(f"Generated: {file_path}")